## Load the Cleaned Dataset

In [7]:
import pandas as pd
df = pd.read_csv(
    "../data/processed/cleaned_demand_timeseries.csv",
    index_col=0,
    parse_dates=True
)

df.head()


C:\Users\ariel\AppData\Local\Temp\ipykernel_14592\4173305874.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


,demand,demand_diff
1,20167,249.0
2,20328,161.0
3,19460,-868.0
4,18654,-806.0
5,18248,-406.0


In [8]:
# Force index to datetime
df.index = pd.to_datetime(df.index)

# Verify
type(df.index)


pandas.core.indexes.datetimes.DatetimeIndex

What I did:
- Loaded the cleaned and preprocessed time-series dataset
- Restored the datetime index for time-based feature creation

Why I did it:
- Feature engineering requires temporal context
- A datetime index enables lag, rolling, and calendar features

Note:
The datetime index was explicitly converted using pd.to_datetime()
to ensure compatibility with time-based feature extraction.


## Creat Lag features

In [9]:
# Previous half hour
df["lag_1"] = df["demand"].shift(1)

# Same time previous day
df["lag_48"] = df["demand"].shift(48)

df[["demand", "lag_1", "lag_48"]].head(60)

,demand,lag_1,lag_48
1970-01-01 00:00:00.000000001,20167,NaN,NaN
1970-01-01 00:00:00.000000002,20328,20167.0,NaN
1970-01-01 00:00:00.000000003,19460,20328.0,NaN
1970-01-01 00:00:00.000000004,18654,19460.0,NaN
1970-01-01 00:00:00.000000005,18248,18654.0,NaN
1970-01-01 00:00:00.000000006,17855,18248.0,NaN
1970-01-01 00:00:00.000000007,17367,17855.0,NaN
1970-01-01 00:00:00.000000008,16774,17367.0,NaN
1970-01-01 00:00:00.000000009,16489,16774.0,NaN
1970-01-01 00:00:00.000000010,16289,16489.0,NaN


What I did:
- Created lagged demand features at t−1 and t−48

Why I did it:
- Electricity demand depends strongly on recent values
- Daily cycles are common in energy consumption patterns


## Create Rolling Statistics

In [10]:
# Rolling mean (daily window)
df["rolling_mean_48"] = df["demand"].rolling(window=48).mean()

df[["demand", "rolling_mean_48"]].head(100)


,demand,rolling_mean_48
1970-01-01 00:00:00.000000001,20167,NaN
1970-01-01 00:00:00.000000002,20328,NaN
1970-01-01 00:00:00.000000003,19460,NaN
1970-01-01 00:00:00.000000004,18654,NaN
1970-01-01 00:00:00.000000005,18248,NaN
...,...,...
1970-01-01 00:00:00.000000096,25564,29080.666667
1970-01-01 00:00:00.000000097,25674,29153.604167
1970-01-01 00:00:00.000000098,25355,29227.541667
1970-01-01 00:00:00.000000099,24800,29302.958333


What I did:
- Computed a rolling mean over a 48-period window

Why I did it:
- Rolling statistics help models identify local trends
- They reduce short-term volatility in the signal


## Extract Date & Time Components

In [11]:
# Critical features for XGBoost and LSTM
df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["month"] = df.index.month

df[["hour", "day_of_week", "month"]].head()


,hour,day_of_week,month
1970-01-01 00:00:00.000000001,0,3,1
1970-01-01 00:00:00.000000002,0,3,1
1970-01-01 00:00:00.000000003,0,3,1
1970-01-01 00:00:00.000000004,0,3,1
1970-01-01 00:00:00.000000005,0,3,1


What I did:
- Extracted hour of day, day of week, and month features

Why I did it:
- Electricity demand varies by time of day and season
- Tree-based and neural models cannot infer this automatically


## Drop rows with NaNs (Created by Lags)

In [12]:
df_fe = df.dropna()

df_fe.isna().sum()

demand             0
demand_diff        0
lag_1              0
lag_48             0
rolling_mean_48    0
hour               0
day_of_week        0
month              0
dtype: int64

What I did:
- Removed rows with missing values caused by lag and rolling features

Why I did it:
- Machine learning models require complete feature vectors
- The data loss is minimal relative to the dataset size

## Save Feature-Engineered Dataset

In [13]:
df_fe.to_csv("../data/processed/feature_engineered_dataset.csv")

df_fe.head()

,demand,demand_diff,lag_1,lag_48,rolling_mean_48,hour,day_of_week,month
1970-01-01 00:00:00.000000049,22173,724.0,21449.0,20167.0,23569.395833,0,3,1
1970-01-01 00:00:00.000000050,21806,-367.0,22173.0,20328.0,23600.187500,0,3,1
1970-01-01 00:00:00.000000051,21180,-626.0,21806.0,19460.0,23636.020833,0,3,1
1970-01-01 00:00:00.000000052,20877,-303.0,21180.0,18654.0,23682.333333,0,3,1
1970-01-01 00:00:00.000000053,20476,-401.0,20877.0,18248.0,23728.750000,0,3,1
